# Data Pipeline Test Notebook

This notebook demonstrates how to:
1. Fetch price data for assets
2. Compute spread, hedge ratio, and z-score
3. Visualize the results

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pairs_trading.data.fetcher import fetch_pair_data, fetch_price_data
from pairs_trading.data.schemas import Asset, Pair
from pairs_trading.data.spread import compute_spread

## 1. Fetch Single Asset Data

In [ ]:
# Fetch S&P 500 data
asset = Asset(symbol="^GSPC", name="S&P 500")

data = fetch_price_data(
    asset=asset,
    start_date=datetime(2023, 1, 1),
    end_date=datetime(2024, 1, 1),
)

print(f"Symbol: {data.symbol}")
print(f"Source: {data.source}")
print(f"Data points: {len(data.df)}")
print(f"\nFirst 5 rows:")
data.df.head()

## 2. Fetch Pair Data

In [ ]:
# Define a pair
asset_a = Asset(symbol="MSFT", name="Microsoft")
asset_b = Asset(symbol="AAPL", name="Apple")
pair = Pair(asset_a=asset_a, asset_b=asset_b)

print(f"Pair ID: {pair.pair_id}")

# Fetch aligned data
start = datetime(2022, 1, 1)
end = datetime(2024, 1, 1)

data_a, data_b = fetch_pair_data(asset_a, asset_b, start, end)

print(f"\n{asset_a.symbol}: {len(data_a.df)} data points")
print(f"{asset_b.symbol}: {len(data_b.df)} data points")

## 3. Compute Spread and Z-Score

In [ ]:
# Compute spread data
spread_data = compute_spread(pair, data_a, data_b, zscore_lookback=20)

print(f"Pair: {spread_data.pair.pair_id}")
print(f"Hedge Ratio: {spread_data.hedge_ratio:.4f}")
print(f"Half-Life: {spread_data.half_life:.2f} days")
print(f"\nZ-Score Statistics:")
print(f"  Min: {spread_data.z_score.min():.2f}")
print(f"  Max: {spread_data.z_score.max():.2f}")

## 4. Visualize Z-Score

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

spread_data.z_score.plot(ax=ax, title=f"Z-Score: {pair.pair_id}")
ax.axhline(0, color="black", linewidth=1)
ax.axhline(2, color="red", linestyle="--", label="+2")
ax.axhline(-2, color="red", linestyle="--", label="-2")
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend()

plt.tight_layout()
plt.show()

## 5. Trading Signals

In [ ]:
z = spread_data.z_score.dropna()

long_signals = (z < -2).sum()
short_signals = (z > 2).sum()

print(f"Total trading days: {len(z)}")
print(f"Long spread signals (z < -2): {long_signals}")
print(f"Short spread signals (z > 2): {short_signals}")